In [ ]:
import json
import math
import os
import random
import re
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from lxml import etree
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

In [ ]:
LR = 5e-5
BETA = 0.005
EMA_GAMMA = 0.95  # Baseline decay
KWORDS = 20  # Target keywords
LABELS = ["positive", "negative", "neutral"]
BATCH_SIZE = 16
GRAD_ACCUM = 2
policy_id = "google/flan-t5-base"
reward_id = "meta-llama/Llama-3.2-3B"
MAX_EPOCHS = 30

# Generation parameters
MAX_NEW_TOKENS = 20
MIN_NEW_TOKENS = 8
MAX_LENGTH = 256
USE_BEAM_SEARCH = True
NUM_BEAMS = 4

# Temperature/sampling
TEMP = 0.7
TOP_P = 0.9

# Reward
REWARD_TYPE = "margin"  # margin only (no blending)
KL_GAMMA = 0.0  # Disabled (was 1e-3, adds noise)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print(f"Generation: MAX_NEW={MAX_NEW_TOKENS}, MIN_NEW={MIN_NEW_TOKENS}")
print(f"Reward: {REWARD_TYPE} | LR={LR} | BETA={BETA}\n")

In [ ]:
def load_absa_xml(xml_path):
    """Load financial ABSA dataset from XML."""
    xml = etree.parse(str(xml_path))
    rows = []
    for sent in xml.xpath("//sentence"):
        sid = sent.get("id")
        text = sent.xpath("./text")[0].text
        for at in sent.xpath(".//aspectTerm"):
            term = at.get("term")
            pol = at.get("polarity").lower()
            rows.append(
                {"id": int(sid), "headline": text, "aspect": term, "label": pol}
            )
    return pd.DataFrame(rows)


class ABSADataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
        self.label2id = {l: i for i, l in enumerate(LABELS)}

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        r = self.frame.iloc[i]
        return {
            "headline": r.headline,
            "aspect": r.aspect,
            "label_id": int(self.label2id[r.label]),
        }


# Load data
XML_PATH = "finentity.xml"
if Path(XML_PATH).exists():
    df = load_absa_xml(XML_PATH)
else:
    raise FileNotFoundError(f"Please provide {XML_PATH}")

full_ds = ABSADataset(df)
n_val = max(50, int(0.15 * len(full_ds)))
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(0)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✓ Data: train={n_train}, val={n_val}\n")

In [ ]:
# Policy: T5 (TRAINABLE - generates keywords)
policy_tok = AutoTokenizer.from_pretrained(policy_id)
policy = AutoModelForSeq2SeqLM.from_pretrained(policy_id).to(device)
policy.train()

# Reward: LLaMA (FROZEN - classifies sentiment)
reward_tok = AutoTokenizer.from_pretrained(reward_id, use_fast=True)
reward_m = AutoModelForCausalLM.from_pretrained(reward_id).to(device)
reward_m.eval()
for p in reward_m.parameters():
    p.requires_grad_(False)

# Config
policy.config.eos_token_id = policy_tok.eos_token_id or 1
policy.config.pad_token_id = policy_tok.pad_token_id or 0
policy.config.decoder_start_token_id = policy.config.pad_token_id

if reward_tok.pad_token_id is None:
    reward_tok.pad_token_id = reward_tok.eos_token_id
reward_m.config.pad_token_id = reward_tok.pad_token_id

print(f"✓ Models loaded")
print(f"  Policy: {policy_id} (TRAINABLE)")
print(f"  Reward: {reward_id} (FROZEN)\n")

In [ ]:
def policy_prompt_t5(headline: str, aspect: str, k: int = 5) -> str:
    """Improved T5 prompt for financial keyword extraction with few-shot examples."""
    return (
        f"You are a financial analyst extracting sentiment indicators for aspect-based analysis.\\n"
        f"Extract up to {k} concise, sentiment-bearing keyword phrases about {{ASPECT}} from {{HEADLINE}}.\\n"
        f"Focus on words/phrases that directly indicate positive, negative, or neutral sentiment.\\n"
        f"Return keywords separated by ' | '.\\n\\n"
        f"Examples:\\n"
        f"ASPECT: Apple | HEADLINE: Apple's stock surged on record earnings beat and strong guidance\\n"
        f"→ earnings beat | record | strong guidance | surged\\n\\n"
        f"ASPECT: Tesla | HEADLINE: Tesla missed Q4 targets, production delayed due to supply chain issues\\n"
        f"→ missed targets | production delayed | supply chain issues\\n\\n"
        f"ASPECT: Microsoft | HEADLINE: Microsoft announced robust cloud growth and expanded AI initiatives\\n"
        f"→ robust growth | expanded | AI initiatives\\n\\n"
        f"ASPECT: {aspect}\\n"
        f"HEADLINE: {headline}\\n"
        f"Keywords:"
    )

In [ ]:
def reward_prompt(headline: str, aspect: str, keywords) -> str:
    """Improved LLaMA prompt with Chain-of-Thought reasoning (FULL VERSION - BEST ACCURACY)."""
    kws = ", ".join(keywords) if isinstance(keywords, list) else str(keywords)

    return (
        "You are a financial sentiment analyst performing aspect-based sentiment analysis.\\n"
        "Analyze the sentiment of a specific ASPECT in a financial headline.\\n\\n"
        "TASK: Let's think step by step:\\n"
        "1) IDENTIFY the aspect being evaluated\\n"
        "2) RECOGNIZE sentiment indicators (provided as keywords)\\n"
        "3) ASSESS the headline context and how it affects the aspect\\n"
        "4) DETERMINE the overall sentiment\\n\\n"
        "SENTIMENT DEFINITIONS (Financial Context):\\n"
        "- POSITIVE: Indicates improvement, growth, success, or favorable developments\\n"
        "  Examples: beat, growth, surge, raised, approved, profit, expansion, upgrade\\n"
        "- NEGATIVE: Indicates decline, loss, failure, or adverse developments\\n"
        "  Examples: miss, cut, decline, loss, layoff, downgrade, recall, lawsuit\\n"
        "- NEUTRAL: Ambiguous, mixed signals, or descriptive without clear direction\\n"
        "  Examples: unchanged, announced, ongoing, considering\\n\\n"
        f"ASPECT: {aspect}\\n"
        f"HEADLINE: {headline}\\n"
        f"Sentiment Keywords (hints): {kws}\\n\\n"
        "REASONING PROCESS:\\n"
        "Step 1 - Focus on the ASPECT (ignore unrelated entities)\\n"
        "Step 2 - Identify what the keywords suggest\\n"
        "Step 3 - Read the headline for the complete context\\n"
        "Step 4 - Determine if the aspect is positively/negatively affected\\n\\n"
        "FINAL ANSWER (respond with ONLY one word):\\n"
        "positive OR negative OR neutral\\n\\n"
        "Answer: "
    )

In [ ]:
def decode_keywords(text: str, k: int = 5, verbose: bool = False) -> list:
    if verbose:
        print(f"  Raw text: '{text}'")

    # Split by pipe
    parts = [p.strip() for p in re.split(r"\s*\|\s*", text) if p.strip()]

    # If only 1 part, try comma/semicolon
    if len(parts) <= 1:
        parts = [p.strip() for p in re.split(r"[;,]", text) if p.strip()]

    # If still empty, split by spaces
    if len(parts) <= 1:
        parts = [p.strip() for p in text.split() if p.strip()]

    if verbose:
        print(f"  Parts after split: {parts}")

    out, seen = [], set()

    for p in parts:
        # prproc
        p = re.sub(r"\s+", " ", p.lower()).strip()
        p = re.sub(r"^[^\w]+|[^\w]+$", "", p)

        if not (1 <= len(p) <= 50):
            continue

        # Truncate to 3 words max
        words = p.split()
        if len(words) > 3:
            p = " ".join(words[:3])

        if len(p) < 2:
            continue

        if p not in seen:
            seen.add(p)
            out.append(p)

        if len(out) >= k:
            break
    return out

In [ ]:
def build_verbalizer_tokens():
    """Map labels to token IDs."""
    verbalizers = {0: "negative", 1: "neutral", 2: "positive"}
    ids = []
    for cls in sorted(verbalizers.keys()):
        text = verbalizers[cls]
        toks = reward_tok.encode(text, add_special_tokens=False)
        if not toks:
            raise ValueError(f"Verbalizer '{text}' is empty")
        ids.append(toks[0])
    return torch.tensor(ids, dtype=torch.long).to(device)


LABEL_TOKEN_IDS = build_verbalizer_tokens()


@torch.no_grad()
def classify_reward(prompt: str):
    """Get LLaMA classification for a single sample."""
    tok = reward_tok(prompt, return_tensors="pt").to(device)
    out = reward_m(**tok)
    logits = out.logits[0, -1, :]
    class_logits = logits[LABEL_TOKEN_IDS]
    pred_id = int(class_logits.argmax().item())
    return pred_id


@torch.no_grad()
def margin_reward_batch(prompt_texts, gold_ids, max_length=512):
    """
    Margin reward: log p(gold) - max_{other} log p(other)
    Batch z-scored for stability.
    """
    tok = reward_tok(
        prompt_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    ).to(device)

    out = reward_m(**tok)
    logits = out.logits
    attn = tok["attention_mask"]

    last_pos = attn.sum(dim=1).clamp(min=1) - 1
    b_idx = torch.arange(logits.size(0), device=device)
    next_logits = logits[b_idx, last_pos, :]

    class_logits = next_logits.index_select(1, LABEL_TOKEN_IDS)
    logp = F.log_softmax(class_logits, dim=1)

    labels = torch.tensor(gold_ids, device=device, dtype=torch.long)

    gold_logp = logp.gather(1, labels.unsqueeze(-1)).squeeze(-1)
    other_logp = logp.clone()
    other_logp[torch.arange(len(gold_ids)), labels] = -float("inf")
    max_other_logp = other_logp.max(dim=1)[0]

    margin = gold_logp - max_other_logp

    # Z-score normalize
    margin = (margin - margin.mean()) / (margin.std() + 1e-8)
    return margin

In [ ]:
@torch.no_grad()
def sample_keywords_batch(headlines, aspects, kwords=5, mode='train'):
    prompts = [policy_prompt_t5(h, a, k=kwords) for h, a in zip(headlines, aspects)]
    enc = policy_tok(prompts, return_tensors="pt", padding=True, truncation=True,
                     max_length=MAX_LENGTH).to(device)

    if mode == 'train':
        # Training: use beam search for better quality
        out = policy.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            min_new_tokens=MIN_NEW_TOKENS,
            do_sample=True,  # ← Changed to False (use beam search)
            top_p=TOP_P,
            # num_beams=NUM_BEAMS if USE_BEAM_SEARCH else 1,  # ← NEW
            no_repeat_ngram_size=2,
            length_penalty=0.5,  # ← Changed from 1.05 (don't penalize length)
            pad_token_id=policy.config.pad_token_id,
            eos_token_id=policy.config.eos_token_id,
        )
    else:  # eval
        # Evaluation: deterministic beam search
        out = policy.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            min_new_tokens=MIN_NEW_TOKENS,
            do_sample=False,  # Same as train
            top_p=TOP_P,
            # num_beams=NUM_BEAMS if USE_BEAM_SEARCH else 1,  # Same as train
            no_repeat_ngram_size=2,
            length_penalty=0.5,  # Changed from 1.1 (consistent)
            pad_token_id=policy.config.pad_token_id,
            eos_token_id=policy.config.eos_token_id,
        )

    ds = policy.config.decoder_start_token_id
    gen_ids_list, kws_list = [], []
    for i in range(out.size(0)):
        seq = out[i]
        if seq.numel() and ds is not None and seq[0].item() == ds:
            seq = seq[1:]
        gen_ids_list.append(seq)
        text = policy_tok.decode(seq, skip_special_tokens=True)
        kws_list.append(decode_keywords(text, k=kwords))

    return gen_ids_list, kws_list, enc


def batch_score_logp_entropy(enc, gen_ids_list):
    """Score sampled keywords (teacher-forced)."""
    pad = policy.config.pad_token_id

    labels = nn.utils.rnn.pad_sequence(
        gen_ids_list, batch_first=True, padding_value=pad
    ).to(device)
    labels_mask = (labels != pad).long()

    decoder_input_ids = policy._shift_right(labels.clone())

    out = policy(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        decoder_input_ids=decoder_input_ids,
        use_cache=False,
    )

    logits = out.logits[:, : labels.size(1), :]
    logp = F.log_softmax(logits, dim=-1)
    tok_lp = logp.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
    tok_lp = tok_lp * labels_mask

    sum_logprob = tok_lp.sum(dim=1)

    p = logp.exp()
    tok_ent = -(p * logp).sum(-1)
    denom = labels_mask.sum(dim=1).clamp_min(1)
    entropy = (tok_ent * labels_mask).sum(dim=1) / denom

    return sum_logprob, entropy

In [ ]:
class EMA:
    def __init__(self, alpha=0.1, init_mean=0.0, init_var=1.0):
        self.alpha = alpha
        self.mean = init_mean
        self.var = init_var
        self.ready = False

    def update(self, x):
        x = float(x)
        if not self.ready:
            self.mean = x
            self.var = 1.0
            self.ready = True
            return
        prev = self.mean
        self.mean = (1 - self.alpha) * self.mean + self.alpha * x
        self.var = (1 - self.alpha) * self.var + self.alpha * (x - prev) ** 2


ema_baseline = EMA(alpha=0.1)

In [ ]:
def train_epoch(epoch_num):
    """Train one epoch."""
    policy.train()
    reward_m.eval()

    running_loss = 0.0
    correct, total = 0, 0
    optim.zero_grad(set_to_none=True)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch_num}", leave=True)

    for step, batch in enumerate(pbar):
        headlines = batch["headline"]
        aspects = batch["aspect"]
        gold_ids = [int(x) for x in batch["label_id"]]

        # Sample keywords
        gen_ids_list, kws_list, enc = sample_keywords_batch(
            headlines, aspects, kwords=KWORDS, mode="train"
        )

        # Score actions
        sum_logp, entropy = batch_score_logp_entropy(enc, gen_ids_list)
        lengths = torch.tensor(
            [max(1, g.numel()) for g in gen_ids_list],
            device=sum_logp.device,
            dtype=sum_logp.dtype,
        )
        avg_logp = (sum_logp / lengths).clamp(min=-20.0, max=0.0)

        # Get keywords and reward prompts
        rp_list = []
        preds = []
        for h, a, kws in zip(headlines, aspects, kws_list):
            rp = reward_prompt(h, a, kws)
            rp_list.append(rp)
            pred_id = classify_reward(rp)
            preds.append(pred_id)

        # Compute reward
        rewards = margin_reward_batch(rp_list, gold_ids)

        # Baseline and advantage
        r_mean = float(rewards.mean().item())
        ema_baseline.update(r_mean)
        advantage = (rewards - ema_baseline.mean) / math.sqrt(ema_baseline.var + 1e-6)
        advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)

        # Policy gradient + entropy
        # loss_pg = -(advantage * avg_logp).mean()
        # loss_ent = -BETA * entropy.mean()
        # loss = (loss_pg + loss_ent) / GRAD_ACCUM

        # Policy gradient (maximize advantage * log prob)
        loss_pg = (advantage * avg_logp).mean()
        # Entropy
        loss_ent = entropy.mean()
        # # Blend with clear weights
        loss = -(loss_pg + 0.02 * loss_ent) # 2% of total

        # Backward with accumulation
        loss.backward()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
            optim.step()
            optim.zero_grad(set_to_none=True)
            if "sched" in globals() and sched is not None:
                sched.step()

        correct += sum(1 for p, g in zip(preds, gold_ids) if p == g)
        total += len(gold_ids)
        running_loss += loss.item()

        pbar.set_postfix(
            {
                "loss": f"{loss.item():.4f}",
                "pg": f"{loss_pg.item():.4f}",
                "ent": f"{loss_ent.item():.4f}",
                "acc": f"{correct / total:.3f}" if total else "0",
            }
        )

    avg_loss = running_loss / max(1, len(train_loader))
    acc = correct / total if total else 0.0

    return avg_loss, acc


@torch.no_grad()
def validate():
    """Validate with deterministic generation."""
    policy.eval()
    reward_m.eval()

    g_all, p_all = [], []

    for batch in tqdm(val_loader, desc="Validating", leave=False):
        headlines = batch["headline"]
        aspects = batch["aspect"]
        gold_ids = [int(x) for x in batch["label_id"]]

        # Deterministic keywords
        _, kws_list, _ = sample_keywords_batch(
            headlines, aspects, kwords=KWORDS, mode="eval"
        )

        for h, a, kws, g in zip(headlines, aspects, kws_list, gold_ids):
            rp = reward_prompt(h, a, kws)
            pred_id = classify_reward(rp)
            g_all.append(g)
            p_all.append(pred_id)

    macro_f1 = f1_score(g_all, p_all, average="macro", zero_division=0)
    micro_f1 = f1_score(g_all, p_all, average="micro", zero_division=0)

    return macro_f1, micro_f1

In [ ]:
if __name__ == "__main__":
    optim = torch.optim.AdamW(policy.parameters(), lr=LR, weight_decay=0.01)

    # Scheduler
    total_steps = len(train_loader) * MAX_EPOCHS // GRAD_ACCUM
    warmup_steps = max(50, int(0.05 * total_steps))
    sched = get_cosine_schedule_with_warmup(optim, warmup_steps, total_steps)

    best_macro = 0.0
    patience, bad = 5, 0

    print(f"Total steps: {total_steps} | Warmup: {warmup_steps}")
    print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}\n")

    # Quick keyword inspection on first batch
    print("="*60)
    print("SAMPLE KEYWORDS (First batch)")
    print("="*60)
    batch = next(iter(train_loader))
    _, kws_list, _ = sample_keywords_batch(batch["headline"], batch["aspect"], KWORDS, mode='eval')
    for h, a, kws in zip(batch["headline"][:2], batch["aspect"][:2], kws_list[:2]):
        print(f"\nHeadline: {h[:60]}...")
        print(f"Aspect: {a}")
        print(f"Keywords ({len(kws)}): {kws}")
    print("="*60 + "\n")

    for ep in range(1, MAX_EPOCHS + 1):
        train_loss, train_acc = train_epoch(ep)
        macro_f1, micro_f1 = validate()

        print(
            f"[Epoch {ep}] train_loss={train_loss:.4f} train_acc={train_acc:.3f} "
            f"VAL macro={macro_f1:.3f} micro={micro_f1:.3f}"
        )

        if macro_f1 > best_macro:
            best_macro = macro_f1
            torch.save(policy.state_dict(), "best_policy.pt")
            print(f"Saved best checkpoint (macro_f1={macro_f1:.3f})")
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print(f"Early stopping (patience={patience})")
                break

    print(f"\n✓ Training complete! Best macro F1: {best_macro:.3f}")